# Modeling Part 2 - The Shiu et al. fly brain model

In Part 1, we built small networks of leaky integrate-and-fire (LIF) neurons in Brian 2. Shiu et al. (2024, *Nature* 634, 210–219) used exactly these ingredients to model the **entire fly brain**: one LIF neuron for each of the ~140,000 neurons in FlyWire, connected by synapses whose weights come from the connectome. When they activated the sugar-sensing gustatory receptor neurons (GRNs) in the model, MN9, the motor neuron that drives proboscis extension, became active, just as in a real fly tasting sugar.

In this notebook we rebuild their model (the code in [`model.py`](https://github.com/philshiu/Drosophila_brain_model/blob/main/model.py)) step by step:
1. load the sugar → MN9 circuit you extracted in FlyWire intro Part 3,
2. create the neurons, synapses and stimulation of the model,
3. stimulate the sugar GRNs and see whether MN9 responds.

In Part 3, we'll use the model for experiments: changing the stimulation frequency, silencing neurons and (optionally) simulating the whole brain.

Gaps for you to fill in are marked `___`.

## 0. Setup
As in part 1, [click here](https://colab.research.google.com/github/ryanrahy/fcnc_workshop/blob/master/day2/modeling_part2_shiu.ipynb) to run this notebook on Colab. If you are running this notebook on Google Colab (or any fresh Python environment), run the next cell to install Brian 2.

In [ ]:
# get code + data
!git clone https://github.com/ryanrahy/fcnc_workshop.git
import os
os.chdir('./fcnc_workshop/day2/')

# install brian
!pip install brian2

In [ ]:
# import packages
from textwrap import dedent   # used in the parameter dictionary below
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from brian2 import *
%matplotlib inline

# our circuit is small, so plain NumPy is fast enough (see Part 3 for the whole brain)
prefs.codegen.target = 'numpy'

## 1. The circuit

At the end of FlyWire intro Part 3, you saved the sugar → MN9 circuit in two files. Put both of them in the same folder as this notebook:
* `sugar_circuit_neurons.csv`: one row per neuron, with its root ID, name, layer (0 = sugar GRNs, 1 and 2 = interneurons, 3 = MN9), predicted neurotransmitter and sign,
* `sugar_circuit_signed_adjacency.csv`: the signed adjacency matrix `W`. `W[i, j]` is the number of synapses from neuron `i` (row) to neuron `j` (column), negative if neuron `i` is inhibitory. Connections with fewer than 5 synapses are 0.

Brian refers to neurons by their **index** (0, 1, ..., N−1) in a `NeuronGroup`, not by their FlyWire root ID. We use the order of `W`: neuron `k` of our model is row `k` (and column `k`) of `W`. The two files don't list the neurons in the same order, so we sort the neuron table to match `W`.

In [ ]:
# read in network connectivity info
W = pd.read_csv('../day1/sugar_circuit_signed_adjacency.csv', index_col=0)   # rows: presynaptic, columns: postsynaptic
W.columns = W.columns.astype('int64')   # the column names (root IDs) are read as text
W.iloc[:5,:5]

In [ ]:
neurons = pd.read_csv('../day1/sugar_circuit_neurons.csv').set_index('root_id').loc[W.index]   # same order as W

# Brian indices of the sugar GRNs (layer 0) and the two MN9s (layer 3)
sugar_idx = np.flatnonzero(neurons.layer == 0)
mn9_idx = np.flatnonzero(neurons.layer == 3)

print(f'{len(neurons)} neurons, {(W != 0).values.sum()} connections, '
      f'{len(sugar_idx)} sugar GRNs, {len(mn9_idx)} MN9 neurons')
neurons.head()

These are the parameters of the Shiu et al model, organized in a dictionary. We've already seen most of them in Part 1 under different names:

| Shiu et al. | Part 1 | value |
|---|---|---|
| `v_0` | resting potential `E_L` | −52 mV |
| `v_th` | threshold `V_th` | −45 mV |
| `v_rst` | reset value (we used `E_L`) | −52 mV |
| `t_mbr` | membrane time constant `tau_m` | 20 ms |
| `t_rfc` | refractory period | 2.2 ms |

The new ones:
* **`g` and `tau`:** instead of making `v` jump, an incoming spike increases the synaptic input `g`, which then decays with time constant `tau` = 5 ms. Meanwhile, `g` pushes `v` up (or down, if it is negative), just like $R_mI$ did in Part 1. This is the decaying synapse from the end of Part 1's connections section.
* **`t_dly`:** a spike takes 1.8 ms to arrive at the postsynaptic neuron (the synaptic `delay`).
* **`w_syn`:** the weight of a single synapse, 0.275 mV. A connection made of `n` synapses has weight `n × w_syn`, and a negative weight if the presynaptic neuron is inhibitory. This is the only *free* parameter: all others come from experiments (see the references in the comments).
* **`r_poi` and `f_poi`:** the stimulation. The sugar GRNs receive Poisson spikes at `r_poi` = 150 Hz, each worth `w_syn × f_poi` = 68.75 mV, enough to make the GRN fire right away (like the "optogenetic" stimulation in Part 1). `r_poi2` is for stimulating a second set of neurons, which we won't use.
* **`rfc`:** the refractory period is a neuron *parameter*, so that the stimulated neurons can have none.
* **`t_run` and `n_run`:** Shiu et al. simulate 1 s, and repeat every simulation 30 times to average out the randomness of the Poisson input. We'll do a single run.

(The `w = 0` in the reset `eq_rst` does nothing, since our neurons have no variable `w`. It's a leftover in the original code.)

In [ ]:
params = {
    # trials
    't_run'     : 1000 * ms,              # duration of trial
    'n_run'     : 30,                     # number of runs

    # network constants
    # Kakaria and de Bivort 2017 https://doi.org/10.3389/fnbeh.2017.00008
    'v_0'       : -52 * mV,               # resting potential
    'v_rst'     : -52 * mV,               # reset potential after spike
    'v_th'      : -45 * mV,               # threshold for spiking
    't_mbr'     :  20 * ms,               # membrane time scale (capacitance * resistance = .002 * uF * 10. * Mohm)

    # Jürgensen et al https://doi.org/10.1088/2634-4386/ac3ba6
    'tau'       : 5 * ms,                 # time constant 

    # Lazar et al https://doi.org/10.7554/eLife.62362
    't_rfc'     : 2.2 * ms,               # refractory period

    # Paul et al 2015 doi: 10.3389/fncel.2015.00029
    't_dly'     : 1.8*ms,                 # delay for changes in post-synaptic neuron

    # Free parameter 
    'w_syn'     : .275 * mV,              # weight per synapse (note: modulated by exponential decay)
    # Default activation rates 
    'r_poi'     : 150*Hz,                 # default rate of the Poisson inputs
    'r_poi2'    :   0*Hz,                 # default rate of a 2nd class of Poisson inputs
    'f_poi'     : 250,                    # scaling factor for Poisson synapse; 250 is sufficient to cause spiking

    # equations for neurons               # alpha synapse https://doi.org/10.1017/CBO9780511815706; See https://brian2.readthedocs.io/en/stable/user/converting_from_integrated_form.html
    'eqs'       : dedent(''' 
                    dv/dt = (v_0 - v + g) / t_mbr : volt (unless refractory)
                    dg/dt = -g / tau               : volt (unless refractory) 
                    rfc                            : second
                    '''),
    # condition for spike
    'eq_th'     : 'v > v_th', 
    # rules for spike        
    'eq_rst'    : 'v = v_rst; w = 0; g = 0 * mV', 
}


We now need to create the neurons and synapses we need for our model

## 2. Building the model

### 2.1 Neurons
Create a single `NeuronGroup` with one neuron for every row of `W`, using the equations (`params['eqs']`), threshold (`params['eq_th']`) and reset (`params['eq_rst']`) from the dictionary. Three arguments are new:
* `method='linear'` is another name for `'exact'`.
* `refractory='rfc'`: the refractory period can be given as the name of a neuron parameter. Here it's `rfc`, so every neuron can have its own.
* `namespace=params`: in Part 1, Brian found constants such as `V_th` among the notebook's variables. Here they are in the dictionary, so we tell Brian to look them up there.

Then start every neuron at its resting potential (`v_0`), with no synaptic input (`g`), and with the default refractory period (`t_rfc`). Remember that the values are stored in the dictionary, e.g. `params['v_0']`.

<details><summary>Show solution</summary>

```python
neu = NeuronGroup(len(W), params['eqs'], method='linear',
                  threshold=params['eq_th'], reset=params['eq_rst'],
                  refractory='rfc', namespace=params)
neu.v = params['v_0']
neu.g = 0*mV
neu.rfc = params['t_rfc']
```
</details>

In [ ]:
start_scope()

neu = NeuronGroup(___, params['eqs'], method='linear',
                  threshold=___, reset=___,
                  refractory='rfc', namespace=params)
neu.v = ___
neu.g = 0*mV
neu.rfc = ___

### 2.2 Synapses
All connections of the circuit go from `neu` to `neu`, so a single `Synapses` object holds all of them. Like in Part 1's weights section, every synapse has a weight `w : volt`. The differences: a spike adds `w` to the postsynaptic **`g`** (not `v`), and it arrives after a delay of `params['t_dly']`.

We first need to find the neuron pairs that have synapses between them. There is one synapse for every non-zero entry of `W`. `np.nonzero` gives the row (presynaptic neuron) and column (postsynaptic neuron) of each of them. 

In [ ]:
pre, post = np.nonzero(W.to_numpy())   # row (presynaptic) and column (postsynaptic) of every connection
pre, post

Next, we create the `Synapses` object and connect our neurons according to their IDs in `pre` and `post`. Finally, we set the weights: number of synapses (with sign) × `w_syn`.

**Check:** 2,079 synapses.

<details><summary>Show solution</summary>

```python
syn = Synapses(neu, neu, model='w : volt', on_pre='g_post += w', delay=params['t_dly'])
syn.connect(i=pre, j=post)
```
</details>

In [ ]:
syn = Synapses(___, ___, model = '___', on_pre = '___', delay = ___)
syn.connect(___)
syn.w = W.to_numpy()[pre, post] * params['w_syn']  # signed number of synapses x weight per synapse

print(len(syn), 'synapses')

### 2.3 Stimulating the sugar GRNs
As in Part 1's "Stimulating neurons" section, we give every sugar GRN its own Poisson spike train: a `PoissonGroup` with one Poisson neuron per GRN, firing at `r_poi`, connected one-to-one to the GRNs. Each Poisson spike adds `w_syn × f_poi` directly to the GRN's `v`. Connect Poisson neuron `k` to the `k`-th sugar GRN, i.e. to neuron `sugar_idx[k]` of `neu`.

(Shiu et al. use a `PoissonInput` for each GRN instead, which has the same effect.)

<details><summary>Show solution</summary>

```python
stim_syn.connect(i=np.arange(len(sugar_idx)), j=sugar_idx)
```
`np.arange(len(sugar_idx))` is `[0, 1, ..., 20]`, so Poisson neuron `k` connects to neuron `sugar_idx[k]`.
</details>

In [ ]:
stim = PoissonGroup(len(sugar_idx), rates=params['r_poi'])
stim_syn = Synapses(stim, neu, model='w : volt', on_pre='v_post += w')
stim_syn.connect(i=___, j=___)
stim_syn.w = params['w_syn'] * params['f_poi']

neu.rfc[sugar_idx] = 0*ms   # no refractory period for the stimulated neurons

### 2.4 Running the model
Record the spikes of all neurons and run the model for `t_run` (1 s). This takes a few seconds.

In [ ]:
spike_mon = SpikeMonitor(neu)
run(params['t_run'])
print(spike_mon.num_spikes, 'spikes')

## 3. Results
Which neurons fired, and how fast? We add each neuron's firing rate to the `neurons` table:

In [ ]:
neurons['rate'] = spike_mon.count / params['t_run'] / Hz

print(f'{(neurons.rate > 0).sum()} of {len(neurons)} neurons fired')
print('MN9 firing rates (Hz):', neurons.rate.iloc[mn9_idx].round(1).to_dict())

# the most active neurons, apart from the stimulated GRNs
neurons[neurons.layer > 0].sort_values('rate', ascending=False).head(15)

In [ ]:
# raster plot: the neurons are sorted by layer, so we draw a line between layers
fig, ax = plt.subplots(figsize=(10, 5))
ax.plot(spike_mon.t/ms, spike_mon.i, '.k', ms=1)
for boundary in np.flatnonzero(np.diff(neurons.layer)):
    ax.axhline(boundary + 0.5, c='gray', lw=0.5)
for k in mn9_idx:
    ax.axhspan(k - 0.5, k + 0.5, color='tab:red', alpha=0.3)
ax.set_xlabel('Time (ms)')
ax.set_ylabel('Neuron index (by layer)')
ax.set_title('Sugar stimulation (MN9 in red)');

**Check:** both MN9s fire, one at roughly 80 Hz and the other at roughly 50 Hz, and about half of the 150 neurons fire at all. (Your numbers will differ a little, since the Poisson input is random.)

### Which neurons are active?
Every interneuron in our circuit lies on a path from the sugar GRNs to MN9, yet many of them stay silent. The plot below shows the firing rate of every neuron, grouped by layer and coloured by its sign. (The points are spread out sideways so that they don't overlap.)

In [ ]:
fig, ax = plt.subplots(figsize=(6, 4))
for sign, color, label in [(1, 'tab:red', 'excitatory'), (-1, 'tab:blue', 'inhibitory')]:
    group = neurons[neurons.sign == sign]
    x = group.layer + np.random.uniform(-0.25, 0.25, len(group))   # spread the points out sideways
    ax.scatter(x, group.rate, s=12, color=color, alpha=0.7, label=label)
ax.set_xticks([0, 1, 2, 3], ['GRNs', 'layer 1', 'layer 2', 'MN9'])
ax.set_ylabel('Firing rate (Hz)')
ax.legend()
plt.show()

# fraction of neurons that fired, per layer and sign
pd.crosstab(neurons.layer, neurons.sign.map({1: 'excitatory', -1: 'inhibitory'}),
            values=neurons.rate > 0, aggfunc='mean').round(2)

**Check:** activity thins out with depth. About two thirds of the layer-1 neurons fire, but only about a third of the layer-2 neurons. In layer 2, inhibitory neurons fire only half as often as excitatory ones (roughly 20% vs 45%).

**Think:** why would a neuron that lies on a path from the sugar GRNs stay silent? (Hint: how many of its inputs are active, and with which sign?)

### What drives MN9?
In FlyWire Part 3 (Section 5.5) you found that, within this circuit, MN9 receives more inhibitory than excitatory synapses. And yet the model predicts that sugar *activates* MN9. Now that we know which neurons are active, we can resolve this. For every input neuron of MN9, we compare
* its **anatomical** input: the number of synapses it makes onto MN9 (negative if it is inhibitory),
* its **active** input: that number multiplied by the neuron's firing rate. A neuron that doesn't fire contributes nothing, however many synapses it makes.

In [ ]:
W_np = W.to_numpy()

fig, axes = plt.subplots(1, 2, figsize=(10, 3))
for ax, k in zip(axes, mn9_idx):
    synapses = W_np[:, k]                          # signed synapses from every neuron onto this MN9
    active_input = synapses * neurons.rate.values  # synapses x firing rate
    for row, x in enumerate([synapses, active_input]):
        exc, inh = x[x > 0].sum(), -x[x < 0].sum()
        ax.barh(row, exc / (exc + inh), color='tab:red')
        ax.barh(row, inh / (exc + inh), left=exc / (exc + inh), color='tab:blue')
    ax.set_yticks([0, 1], ['anatomy\n(synapses)', 'activity\n(synapses x rate)'])
    ax.set_xlabel('Fraction of input (red: excitatory, blue: inhibitory)')
    ax.set_title(f'MN9 (neuron {k})')
fig.tight_layout()
plt.show()

# the 10 strongest active inputs to the two MN9s together
inputs = pd.DataFrame({'name': neurons.name, 'top_nt': neurons.top_nt,
                       'synapses onto MN9': W_np[:, mn9_idx].sum(axis=1), 'rate': neurons.rate})
inputs['active input'] = inputs['synapses onto MN9'] * inputs.rate
inputs[inputs['synapses onto MN9'] != 0].sort_values('active input', key=abs, ascending=False).head(10)

**Check:** by synapse count, more than half of MN9's input is inhibitory. Weighted by activity, roughly 70–85% of it is excitatory. A handful of neurons, such as `roundup`, provide most of MN9's active input.

**Think:**
* Most of MN9's inhibitory inputs are silent during sugar stimulation. What does this tell you about predicting the function of a circuit from its wiring diagram alone?
* What do you expect to happen to MN9 if you silence the strongest drivers in the table? You can test your prediction in Part 3.

### References
* Shiu et al. (2024) *A Drosophila computational brain model reveals sensorimotor processing.* Nature 634, 210–219. Code: [github.com/philshiu/Drosophila_brain_model](https://github.com/philshiu/Drosophila_brain_model)
* Stimberg, Brette & Goodman (2019) *Brian 2, an intuitive and efficient neural simulator.* eLife 8, e47314.